# DeepDTA on Davis: browser version (permanent static link)

Run Sections 1 to 11 first (this trains the model). If the training session is still active, you can skip straight to Section 12.

The model runs in the visitor's browser, so the website is a static page. Static Hugging Face Spaces are free.


## 1. Setup

In [ ]:
!pip install -q fastapi httpx huggingface_hub tensorflowjs


## 2. Settings
All Davis values come from the paper. Change a value here and it applies everywhere.

In [ ]:
import os

PROJECT_DIR = "/content/My_DeepDTA"
MODEL_DIR = os.path.join(PROJECT_DIR, "model")
RESULTS_DIR = os.path.join(PROJECT_DIR, "results")
SPACE_DIR = os.path.join(PROJECT_DIR, "space")   # everything published to the permanent link
for folder in [MODEL_DIR, RESULTS_DIR, SPACE_DIR]:
    os.makedirs(folder, exist_ok=True)

# Davis values from the paper
MAX_SMILES_LENGTH = 85        # Section 2.3
MAX_PROTEIN_LENGTH = 1200     # Section 2.3
EMBEDDING_DIM = 128           # Section 2.2
NUM_FILTERS = 32              # 32, 64, 96 filters in the three conv layers (Section 3.3)
SMILES_KERNEL = 4             # filter length for compounds (Table 2 range: 4, 6, 8)
PROTEIN_KERNEL = 8            # filter length for proteins (Table 2 range: 4, 8, 12)
DENSE_UNITS = (1024, 1024, 512)  # Section 2.3
DROPOUT_RATE = 0.1            # Section 2.3
LEARNING_RATE = 0.001         # Section 2.3 (Adam)
BATCH_SIZE = 256              # Section 2.3
EPOCHS = 50                   # chosen for training time (the paper uses 100)
PKD_THRESHOLD = 7.0           # Section 3.4: pKd >= 7 counts as binding (used for AUPR)

print("Project folders ready:", PROJECT_DIR)

## 3. Imports and download the DeepDTA data

In [ ]:
import json
import pickle
from collections import OrderedDict

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import mean_squared_error, average_precision_score

print("TensorFlow:", tf.__version__)

!wget -q https://github.com/hkmztrk/DeepDTA/archive/refs/heads/master.zip -O /content/deepdta.zip
!unzip -q -o /content/deepdta.zip -d /content/deepdta_repo

DAVIS_DIR = "/content/deepdta_repo/DeepDTA-master/data/davis"
print(os.listdir(DAVIS_DIR))

## 4. Load the Davis dataset
Drugs are SMILES strings, targets are protein sequences, and Y holds the Kd values in nM. Missing measurements are NaN.

In [ ]:
with open(os.path.join(DAVIS_DIR, "ligands_can.txt")) as f:
    ligands = json.load(f, object_pairs_hook=OrderedDict)
with open(os.path.join(DAVIS_DIR, "proteins.txt")) as f:
    proteins = json.load(f, object_pairs_hook=OrderedDict)
with open(os.path.join(DAVIS_DIR, "Y"), "rb") as f:
    Y = pickle.load(f, encoding="latin1")

drugs = list(ligands.values())       # SMILES strings, one per drug
targets = list(proteins.values())    # amino-acid sequences, one per protein
Y = np.array(Y, dtype=np.float64)    # Kd in nM

print("Drugs:", len(drugs), "| Proteins:", len(targets))
print("Y shape:", Y.shape)
assert Y.shape == (len(drugs), len(targets)), "Y does not match the drug and protein lists"

## 5. Convert Kd to pKd
Paper Equation (1): pKd = -log10(Kd in molar).

In [ ]:
affinity = -np.log10(Y / 1e9)        # NaN stays NaN
drug_idx, target_idx = np.where(~np.isnan(affinity))

print("Known drug-target pairs:", len(drug_idx))   # paper: 30,056
print("pKd range:", np.nanmin(affinity), "to", np.nanmax(affinity))

## 6. Train and test folds
Paper Section 3.3: the official test fold is held out. All five training folds are used for training.

In [ ]:
with open(os.path.join(DAVIS_DIR, "folds", "train_fold_setting1.txt")) as f:
    train_folds = json.load(f)
with open(os.path.join(DAVIS_DIR, "folds", "test_fold_setting1.txt")) as f:
    test_fold = json.load(f)

train_positions = np.array([p for fold in train_folds for p in fold])
test_positions = np.array(test_fold)

train_drugs = drug_idx[train_positions]
train_targets = target_idx[train_positions]
test_drugs = drug_idx[test_positions]
test_targets = target_idx[test_positions]

y_train = affinity[train_drugs, train_targets]
y_test = affinity[test_drugs, test_targets]

print("Training pairs:", len(y_train))
print("Test pairs:", len(y_test))

## 7. Encode SMILES and protein sequences
Each character becomes a number (0 is padding). The vocabulary is built from the Davis data, so every character in the data is included, including `@` and `\`.

In [ ]:
smiles_chars = sorted(set("".join(drugs)))
protein_chars = sorted(set("".join(targets)))

smiles_vocab = {ch: i + 1 for i, ch in enumerate(smiles_chars)}
protein_vocab = {ch: i + 1 for i, ch in enumerate(protein_chars)}

print("SMILES symbols:", len(smiles_vocab), "| contains '@':", "@" in smiles_vocab, "| contains backslash:", "\\" in smiles_vocab)
print("Protein symbols:", len(protein_vocab))

def encode(sequence, vocab, max_len):
    encoded = np.zeros(max_len, dtype=np.int32)   # zeros = padding
    for i, ch in enumerate(sequence[:max_len]):   # truncate long sequences
        encoded[i] = vocab.get(ch, 0)
    return encoded

encoded_drugs = np.array([encode(s, smiles_vocab, MAX_SMILES_LENGTH) for s in drugs])
encoded_targets = np.array([encode(s, protein_vocab, MAX_PROTEIN_LENGTH) for s in targets])

X_drug_train = encoded_drugs[train_drugs]
X_target_train = encoded_targets[train_targets]
X_drug_test = encoded_drugs[test_drugs]
X_target_test = encoded_targets[test_targets]

print("Drug input shape:", X_drug_train.shape, "| Protein input shape:", X_target_train.shape)

## 8. Build the DeepDTA model
Two CNN branches (one for drugs, one for proteins), joined by fully connected layers. Structure follows Figure 2 and Section 2.3.

In [ ]:
from tensorflow.keras.layers import Input, Embedding, Conv1D, GlobalMaxPooling1D, Concatenate, Dense, Dropout
from tensorflow.keras.models import Model

def cnn_branch(name, max_len, vocab_size, kernel_size):
    inp = Input(shape=(max_len,), name=name + "_input")
    x = Embedding(input_dim=vocab_size + 1, output_dim=EMBEDDING_DIM, name=name + "_embedding")(inp)
    x = Conv1D(NUM_FILTERS, kernel_size, activation="relu")(x)
    x = Conv1D(NUM_FILTERS * 2, kernel_size, activation="relu")(x)
    x = Conv1D(NUM_FILTERS * 3, kernel_size, activation="relu")(x)
    x = GlobalMaxPooling1D()(x)
    return inp, x

drug_input, drug_features = cnn_branch("drug", MAX_SMILES_LENGTH, len(smiles_vocab), SMILES_KERNEL)
protein_input, protein_features = cnn_branch("protein", MAX_PROTEIN_LENGTH, len(protein_vocab), PROTEIN_KERNEL)

x = Concatenate()([drug_features, protein_features])
x = Dense(DENSE_UNITS[0], activation="relu")(x)
x = Dropout(DROPOUT_RATE)(x)
x = Dense(DENSE_UNITS[1], activation="relu")(x)
x = Dropout(DROPOUT_RATE)(x)
x = Dense(DENSE_UNITS[2], activation="relu")(x)
output = Dense(1, activation="linear", name="pkd")(x)

model = Model(inputs=[drug_input, protein_input], outputs=output, name="DeepDTA_Davis")
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE), loss="mse")
model.summary()

## 9. Train the model
Training runs for the number of epochs set in Section 2 (50 here).

In [ ]:
history = model.fit(
    [X_drug_train, X_target_train],
    y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1,
)
print("Training finished.")

## 10. Evaluate and save results
Metrics follow the paper: Concordance Index (Eq. 7-8), MSE (Eq. 2), r²m (Eq. 9) and AUPR.

In [ ]:
def concordance_index(y_true, y_pred):
    # Paper Eq. (7)-(8): share of pairs ranked in the same order; tied predictions count 0.5
    total = 0.0
    pairs = 0
    for i in range(len(y_true)):
        larger = y_true[i] > y_true
        diff = y_pred[i] - y_pred[larger]
        total += np.sum(diff > 0) + 0.5 * np.sum(diff == 0)
        pairs += np.sum(larger)
    return total / pairs

def r2m(y_true, y_pred):
    # Roy et al. r²m metric (paper Eq. 9), using predicted and observed values
    r2 = np.corrcoef(y_true, y_pred)[0, 1] ** 2
    k = np.sum(y_pred * y_true) / np.sum(y_true ** 2)
    r0_2 = 1 - np.sum((y_pred - k * y_true) ** 2) / np.sum((y_pred - y_pred.mean()) ** 2)
    return r2 * (1 - np.sqrt(max(r2 - r0_2, 0.0)))

y_pred = model.predict([X_drug_test, X_target_test], verbose=0).ravel()

ci = concordance_index(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2m_value = r2m(y_test, y_pred)
aupr = average_precision_score((y_test >= PKD_THRESHOLD).astype(int), y_pred)

print("========== DAVIS TEST RESULTS ==========")
print("Concordance Index (CI):", round(ci, 4), " (paper: 0.878)")
print("MSE:                  ", round(mse, 4), " (paper: 0.261)")
print("r²m:                  ", round(r2m_value, 4), " (paper: 0.630)")
print("AUPR:                 ", round(aupr, 4), " (paper: 0.714)")

results = pd.DataFrame([{
    "dataset": "Davis", "epochs": EPOCHS, "CI": ci, "MSE": mse, "r2m": r2m_value, "AUPR": aupr
}])
results.to_csv(os.path.join(RESULTS_DIR, "davis_metrics.csv"), index=False)

predictions = pd.DataFrame({"actual_pKd": y_test, "predicted_pKd": y_pred})
predictions.to_csv(os.path.join(RESULTS_DIR, "davis_predictions.csv"), index=False)

print("\nSaved:", os.path.join(RESULTS_DIR, "davis_metrics.csv"))
print("Saved:", os.path.join(RESULTS_DIR, "davis_predictions.csv"))

## 11. Save the model and its character mapping
The model and the vocabularies are saved together, so the website always uses the same mapping as training.

In [ ]:
import shutil

MODEL_FILE = os.path.join(MODEL_DIR, "deepdta_davis.keras")
CONFIG_FILE = os.path.join(MODEL_DIR, "config.json")

model.save(MODEL_FILE)

config = {
    "max_smiles_length": MAX_SMILES_LENGTH,
    "max_protein_length": MAX_PROTEIN_LENGTH,
    "smiles_vocab": smiles_vocab,
    "protein_vocab": protein_vocab,
}
with open(CONFIG_FILE, "w") as f:
    json.dump(config, f)

# Copy into the Space folder
space_model_dir = os.path.join(SPACE_DIR, "model")
os.makedirs(space_model_dir, exist_ok=True)
shutil.copy(MODEL_FILE, space_model_dir)
shutil.copy(CONFIG_FILE, space_model_dir)

print("Model saved:", MODEL_FILE, round(os.path.getsize(MODEL_FILE) / 1e6, 2), "MB")

## 12. Export the model to the browser format
Converts the trained model into TensorFlow.js format, which runs in the browser.

In [ ]:

import shutil
import subprocess

SAVED_MODEL_DIR = os.path.join(MODEL_DIR, "saved_model")
WEB_DIR = os.path.join(PROJECT_DIR, "web")
WEB_MODEL_DIR = os.path.join(WEB_DIR, "model")

shutil.rmtree(SAVED_MODEL_DIR, ignore_errors=True)
shutil.rmtree(WEB_MODEL_DIR, ignore_errors=True)
os.makedirs(WEB_DIR, exist_ok=True)

model.export(SAVED_MODEL_DIR)

result = subprocess.run(
    [
        "tensorflowjs_converter",
        "--input_format=tf_saved_model",
        "--output_format=tfjs_graph_model",
        SAVED_MODEL_DIR,
        WEB_MODEL_DIR,
    ],
    capture_output=True,
    text=True,
)
print(result.stderr[-1500:])
assert os.path.exists(os.path.join(WEB_MODEL_DIR, "model.json")), "Conversion failed. Send me the messages above."

shutil.copy(CONFIG_FILE, WEB_DIR)
print("Browser model ready. Files:", os.listdir(WEB_MODEL_DIR))

## 13. Website files (browser version)
Writes the HTML, CSS and JavaScript. The JavaScript repeats the same encoding as the Python code (same vocabulary, lengths 85 and 1200, and the same truncation and padding).

In [ ]:
WEB_DIR = os.path.join(PROJECT_DIR, "web")
os.makedirs(WEB_DIR, exist_ok=True)

index_html = r'''
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>DeepDTA | Binding Affinity Prediction</title>
    <link rel="stylesheet" href="style.css">
</head>
<body>

    <header class="navbar">
        <div class="logo">
            <div class="logo-icon">🧬</div>
            <div>
                <div class="logo-title">DeepDTA</div>
                <div class="logo-subtitle">Drug–Target Affinity Prediction</div>
            </div>
        </div>
        <div class="nav-badge"><span class="status-dot"></span>AI MODEL ONLINE</div>
    </header>

    <main class="container">

        <section class="hero">
            <div class="hero-badge">Deep Learning for Drug Discovery</div>
            <h1>Predict Drug–Target <span>Binding Affinity</span></h1>
            <p>
                Enter a drug's SMILES representation and a protein sequence
                to estimate their binding affinity using the DeepDTA
                convolutional neural network.
            </p>
        </section>

        <section class="prediction-card">

            <div class="card-header">
                <div>
                    <div class="section-label">PREDICTION</div>
                    <h2>Drug–Target Analysis</h2>
                </div>
                <div class="model-pill">CNN</div>
            </div>

            <div class="input-grid">
                <div class="input-section">
                    <div class="input-title">
                        <div class="input-icon drug-icon">💊</div>
                        <div>
                            <h3>Drug SMILES</h3>
                            <p>Chemical structure representation</p>
                        </div>
                    </div>
                    <textarea id="smiles" placeholder="Example: CC(C)Cc1ccc(cc1)[C@@H](C)C(=O)O" spellcheck="false"></textarea>
                    <div class="input-hint">Enter the molecular SMILES notation</div>
                </div>

                <div class="input-section">
                    <div class="input-title">
                        <div class="input-icon protein-icon">🧬</div>
                        <div>
                            <h3>Protein Sequence</h3>
                            <p>Amino acid sequence</p>
                        </div>
                    </div>
                    <textarea id="protein" placeholder="Example: MKTIIALSYIFCLVFADYKDDDDK" spellcheck="false"></textarea>
                    <div class="input-hint">Enter the protein amino acid sequence</div>
                </div>
            </div>

            <button id="predictButton" class="predict-button" onclick="predictAffinity()">
                <span id="buttonText">Predict Binding Affinity</span>
                <span class="button-arrow">→</span>
            </button>

            <div class="result-card">
                <div class="result-left">
                    <div class="result-icon">⚡</div>
                    <div>
                        <div class="result-label">PREDICTED BINDING AFFINITY</div>
                        <div class="result-description">DeepDTA model prediction</div>
                    </div>
                </div>
                <div class="result-value">
                    <span id="resultValue">—</span>
                    <small>pKd</small>
                </div>
            </div>

            <div class="scale-section" id="scaleSection" hidden>
                <div class="category-label" id="categoryLabel"></div>
                <div class="scale-bar">
                    <div class="seg weak"></div>
                    <div class="seg moderate"></div>
                    <div class="seg strong"></div>
                    <div class="marker" id="marker"></div>
                </div>
                <div class="scale-labels">
                    <span>Weak (5 to under 6)</span>
                    <span>Moderate (6 to under 7)</span>
                    <span>Strong (7 and above)</span>
                </div>
            </div>

        </section>

        <section class="info-grid">
            <div class="info-card">
                <div class="info-card-icon">🧪</div>
                <h3>Drug Representation</h3>
                <p>The drug is represented using its SMILES molecular notation and processed as a sequence.</p>
            </div>
            <div class="info-card">
                <div class="info-card-icon">🧬</div>
                <h3>Protein Representation</h3>
                <p>Protein amino acid sequences are encoded into numerical representations for the neural network.</p>
            </div>
            <div class="info-card">
                <div class="info-card-icon">🤖</div>
                <h3>Deep Learning</h3>
                <p>Two convolutional neural network branches learn features from drug and protein sequences.</p>
            </div>
        </section>

    </main>

    <footer>
        <div>
            <strong>DeepDTA</strong>
            <span>Drug–Target Binding Affinity Prediction</span>
        </div>
        <div class="footer-right">Powered by Deep Learning</div>
    </footer>

    <script src="https://cdn.jsdelivr.net/npm/@tensorflow/tfjs@4.22.0/dist/tf.min.js"></script>
    <script src="script.js"></script>
</body>
</html>
'''

style_css = r'''
* { margin: 0; padding: 0; box-sizing: border-box; }
[hidden] { display: none !important; }

:root {
    --bg: #07111f;
    --card: rgba(17, 31, 52, 0.88);
    --border: rgba(148, 163, 184, 0.14);
    --text: #f8fafc;
    --muted: #94a3b8;
    --primary: #38bdf8;
    --secondary: #8b5cf6;
    --success: #34d399;
}

body {
    font-family: Inter, system-ui, -apple-system, "Segoe UI", sans-serif;
    background: linear-gradient(180deg, #07111f 0%, #091525 45%, #07111f 100%);
    color: var(--text);
    min-height: 100vh;
}

.navbar {
    width: 100%; max-width: 1180px; margin: auto; padding: 26px 24px;
    display: flex; align-items: center; justify-content: space-between;
}
.logo { display: flex; align-items: center; gap: 13px; }
.logo-icon {
    width: 46px; height: 46px; display: flex; align-items: center; justify-content: center;
    border-radius: 14px; font-size: 24px;
    background: linear-gradient(135deg, rgba(56, 189, 248, 0.2), rgba(139, 92, 246, 0.2));
    border: 1px solid rgba(148, 163, 184, 0.18);
}
.logo-title { font-size: 20px; font-weight: 800; }
.logo-subtitle { font-size: 11px; color: var(--muted); margin-top: 2px; }
.nav-badge {
    display: flex; align-items: center; gap: 8px; padding: 8px 13px; border-radius: 999px;
    background: rgba(52, 211, 153, 0.08); border: 1px solid rgba(52, 211, 153, 0.18);
    color: #86efac; font-size: 10px; font-weight: 700; letter-spacing: 0.8px;
}
.status-dot {
    width: 7px; height: 7px; border-radius: 50%; background: var(--success);
    box-shadow: 0 0 10px rgba(52, 211, 153, 0.8);
}

.container { width: min(1100px, calc(100% - 40px)); margin: auto; }

.hero { text-align: center; max-width: 850px; margin: 60px auto 46px; }
.hero-badge {
    display: inline-block; padding: 8px 14px; border-radius: 999px;
    background: rgba(56, 189, 248, 0.07); border: 1px solid rgba(56, 189, 248, 0.18);
    color: #7dd3fc; font-size: 12px; font-weight: 600;
}
.hero h1 { margin-top: 22px; font-size: clamp(38px, 6vw, 60px); line-height: 1.05; font-weight: 850; }
.hero h1 span {
    display: block; background: linear-gradient(90deg, #38bdf8, #8b5cf6);
    -webkit-background-clip: text; background-clip: text; color: transparent;
}
.hero p { max-width: 690px; margin: 22px auto 0; color: var(--muted); font-size: 16px; line-height: 1.75; }

.prediction-card {
    background: var(--card); border: 1px solid var(--border); border-radius: 26px;
    padding: 30px; box-shadow: 0 30px 80px rgba(0, 0, 0, 0.25);
}
.card-header { display: flex; align-items: center; justify-content: space-between; margin-bottom: 28px; }
.section-label { font-size: 10px; font-weight: 800; letter-spacing: 1.8px; color: var(--primary); margin-bottom: 7px; }
.card-header h2 { font-size: 25px; }
.model-pill {
    padding: 8px 14px; border-radius: 10px; background: rgba(139, 92, 246, 0.1);
    border: 1px solid rgba(139, 92, 246, 0.22); color: #c4b5fd; font-size: 11px; font-weight: 800;
}

.input-grid { display: grid; grid-template-columns: 1fr 1fr; gap: 20px; }
.input-section {
    background: rgba(7, 17, 31, 0.55); border: 1px solid rgba(148, 163, 184, 0.1);
    border-radius: 18px; padding: 20px;
}
.input-title { display: flex; align-items: center; gap: 12px; margin-bottom: 15px; }
.input-icon {
    width: 42px; height: 42px; display: flex; align-items: center; justify-content: center;
    border-radius: 12px; font-size: 20px;
}
.drug-icon { background: rgba(56, 189, 248, 0.1); }
.protein-icon { background: rgba(139, 92, 246, 0.1); }
.input-title h3 { font-size: 15px; font-weight: 700; }
.input-title p { margin-top: 3px; font-size: 11px; color: var(--muted); }
textarea {
    width: 100%; height: 150px; resize: vertical; padding: 16px; border-radius: 13px;
    border: 1px solid rgba(148, 163, 184, 0.12); background: rgba(2, 8, 23, 0.72);
    color: #e2e8f0; outline: none; font-family: Consolas, "Liberation Mono", monospace;
    font-size: 13px; line-height: 1.7;
}
textarea:focus { border-color: rgba(56, 189, 248, 0.45); }
.input-hint { margin-top: 10px; font-size: 10px; color: #64748b; }

.predict-button {
    width: 100%; margin-top: 22px; padding: 16px 20px; border: none; border-radius: 14px;
    cursor: pointer; display: flex; align-items: center; justify-content: center; gap: 12px;
    background: linear-gradient(90deg, #0284c7, #7c3aed); color: white;
    font-size: 14px; font-weight: 750; transition: transform 0.2s ease, opacity 0.2s ease;
}
.predict-button:hover { transform: translateY(-2px); }
.predict-button:disabled { cursor: wait; opacity: 0.65; }
.button-arrow { font-size: 20px; line-height: 0; }

.result-card {
    margin-top: 22px; padding: 20px 22px; border-radius: 17px;
    background: linear-gradient(90deg, rgba(56, 189, 248, 0.07), rgba(139, 92, 246, 0.07));
    border: 1px solid rgba(56, 189, 248, 0.13);
    display: flex; align-items: center; justify-content: space-between;
}
.result-left { display: flex; align-items: center; gap: 13px; }
.result-icon {
    width: 42px; height: 42px; display: flex; align-items: center; justify-content: center;
    border-radius: 12px; background: rgba(52, 211, 153, 0.1); font-size: 18px;
}
.result-label { font-size: 10px; letter-spacing: 1.3px; font-weight: 800; color: #7dd3fc; }
.result-description { margin-top: 4px; color: var(--muted); font-size: 11px; }
.result-value { display: flex; align-items: baseline; gap: 7px; }
.result-value span { font-size: 30px; font-weight: 800; color: #c4b5fd; }
.result-value small { color: var(--muted); font-size: 12px; font-weight: 700; }

.scale-section {
    margin-top: 22px; padding: 22px; border-radius: 17px;
    background: rgba(7, 17, 31, 0.55); border: 1px solid rgba(148, 163, 184, 0.1);
}
.category-label { font-size: 15px; font-weight: 700; margin-bottom: 16px; }
.category-label.weak { color: #fbbf24; }
.category-label.moderate { color: #60a5fa; }
.category-label.strong { color: #34d399; }
.scale-bar { position: relative; display: flex; height: 14px; border-radius: 999px; overflow: visible; }
.seg { height: 100%; }
.seg.weak { flex: 1; background: #f59e0b; border-radius: 999px 0 0 999px; }
.seg.moderate { flex: 1; background: #3b82f6; }
.seg.strong { flex: 4; background: #10b981; border-radius: 0 999px 999px 0; }
.marker {
    position: absolute; top: -6px; left: 0; width: 26px; height: 26px; margin-left: -13px;
    border-radius: 50%; background: white; border: 3px solid #07111f;
    box-shadow: 0 0 0 2px rgba(255, 255, 255, 0.4); transition: left 0.4s ease;
}
.scale-labels { display: flex; justify-content: space-between; margin-top: 12px; font-size: 11px; color: var(--muted); }
.scale-labels span:nth-child(2) { text-align: center; }
.scale-labels span:nth-child(3) { text-align: right; }

.info-grid { display: grid; grid-template-columns: repeat(3, 1fr); gap: 18px; margin: 65px 0 80px; }
.info-card {
    padding: 24px; border-radius: 18px; background: rgba(17, 31, 52, 0.5);
    border: 1px solid rgba(148, 163, 184, 0.1);
}
.info-card-icon { font-size: 24px; margin-bottom: 15px; }
.info-card h3 { font-size: 15px; }
.info-card p { margin-top: 8px; color: var(--muted); font-size: 12px; line-height: 1.7; }

footer {
    border-top: 1px solid rgba(148, 163, 184, 0.08); width: min(1100px, calc(100% - 40px));
    margin: auto; padding: 25px 0 35px; display: flex; justify-content: space-between;
    color: #64748b; font-size: 11px;
}
footer strong { color: #cbd5e1; margin-right: 7px; }
.footer-right { color: #475569; }

@media (max-width: 850px) {
    .input-grid { grid-template-columns: 1fr; }
    .info-grid { grid-template-columns: 1fr; }
}
@media (max-width: 600px) {
    .nav-badge { display: none; }
    .prediction-card { padding: 18px; }
    .result-card { align-items: flex-start; gap: 15px; flex-direction: column; }
    footer { flex-direction: column; gap: 8px; }
}
'''

script_js = r'''
const SMILES_MAX_LENGTH = 85;
const PROTEIN_MAX_LENGTH = 1200;
const SCALE_MIN = 5;
const SCALE_MAX = 11;

let model = null;
let config = null;

async function loadModel() {
    try {
        const configResponse = await fetch("config.json");
        config = await configResponse.json();
        model = await tf.loadGraphModel("model/model.json");
    } catch (error) {
        console.error("Model failed to load:", error);
    }
}

const modelReady = loadModel();

function encode(sequence, vocab, maxLength) {
    const encoded = new Array(maxLength).fill(0);
    const length = Math.min(sequence.length, maxLength);
    for (let i = 0; i < length; i++) {
        const ch = sequence[i];
        encoded[i] = Object.prototype.hasOwnProperty.call(vocab, ch) ? vocab[ch] : 0;
    }
    return encoded;
}

function findInputName(keyword) {
    const input = model.inputs.find(i => i.name.toLowerCase().includes(keyword));
    return input ? input.name : null;
}

async function runModel(smiles, protein) {
    const drugName = findInputName("drug");
    const proteinName = findInputName("protein");
    if (!drugName || !proteinName) {
        throw new Error("Model inputs not found");
    }

    const drugTensor = tf.tensor2d(
        [encode(smiles, config.smiles_vocab, SMILES_MAX_LENGTH)],
        [1, SMILES_MAX_LENGTH],
        "int32"
    );
    const proteinTensor = tf.tensor2d(
        [encode(protein, config.protein_vocab, PROTEIN_MAX_LENGTH)],
        [1, PROTEIN_MAX_LENGTH],
        "int32"
    );

    const output = model.predict({ [drugName]: drugTensor, [proteinName]: proteinTensor });
    const values = await output.data();

    drugTensor.dispose();
    proteinTensor.dispose();
    output.dispose();

    return values[0];
}

function classify(value) {
    if (value >= 7) return { label: "Strong binding", range: "pKd 7 and above", cls: "strong" };
    if (value >= 6) return { label: "Moderate binding", range: "pKd 6 to under 7", cls: "moderate" };
    return { label: "Weak binding", range: "pKd 5 to under 6", cls: "weak" };
}

function showScale(value) {
    const info = classify(value);
    const label = document.getElementById("categoryLabel");
    label.textContent = info.label + " (" + info.range + ")";
    label.className = "category-label " + info.cls;

    const clamped = Math.min(Math.max(value, SCALE_MIN), SCALE_MAX);
    const percent = (clamped - SCALE_MIN) / (SCALE_MAX - SCALE_MIN) * 100;
    document.getElementById("marker").style.left = percent + "%";

    document.getElementById("scaleSection").hidden = false;
}

async function predictAffinity() {
    const smiles = document.getElementById("smiles").value.trim();
    const protein = document.getElementById("protein").value.trim().toUpperCase();
    const result = document.getElementById("resultValue");
    const button = document.getElementById("predictButton");
    const buttonText = document.getElementById("buttonText");

    if (smiles === "") {
        alert("Please enter a Drug SMILES.");
        return;
    }
    if (protein === "") {
        alert("Please enter a Protein Sequence.");
        return;
    }

    result.textContent = "...";
    document.getElementById("scaleSection").hidden = true;
    button.disabled = true;
    buttonText.textContent = "Predicting...";

    try {
        await modelReady;
        if (!model) {
            result.textContent = "Model failed to load. Please refresh the page.";
            return;
        }
        const value = await runModel(smiles, protein);
        result.textContent = value.toFixed(4);
        showScale(value);
    } catch (error) {
        console.error(error);
        result.textContent = "Prediction error.";
    } finally {
        button.disabled = false;
        buttonText.textContent = "Predict Binding Affinity";
    }
}
'''

with open(os.path.join(WEB_DIR, "index.html"), "w") as f:
    f.write(index_html)
with open(os.path.join(WEB_DIR, "style.css"), "w") as f:
    f.write(style_css)
with open(os.path.join(WEB_DIR, "script.js"), "w") as f:
    f.write(script_js)

print("Website files written to", WEB_DIR)


## 14. Expected value for one test pair
Prints the Python prediction for one Davis test pair and saves the input to a file. After publishing, paste the same input into the website. The browser should show the same number.

In [ ]:
sample_smiles = drugs[test_drugs[0]]
sample_protein = targets[test_targets[0]]

python_prediction = float(model.predict([X_drug_test[:1], X_target_test[:1]], verbose=0)[0][0])
print("Python prediction:", round(python_prediction, 4))
print("Actual pKd:", round(float(y_test[0]), 4))

sample_file = os.path.join(RESULTS_DIR, "sample_input.txt")
with open(sample_file, "w") as f:
    f.write(sample_smiles + "\n" + sample_protein + "\n")
print("Sample input saved to:", sample_file)
print("SMILES:", sample_smiles)

## 15. Publish to a permanent link (free static Space)
Uploads the web folder to a Hugging Face Space. Needs the `HF_TOKEN` secret from before.

In [ ]:
from huggingface_hub import HfApi

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN not found. Add it in Colab Secrets and turn on notebook access.")

with open(os.path.join(WEB_DIR, "README.md"), "w") as f:
    f.write(
        "---\n"
        "title: DeepDTA Davis\n"
        "emoji: 🧬\n"
        "colorFrom: blue\n"
        "colorTo: purple\n"
        "sdk: static\n"
        "pinned: false\n"
        "---\n\n"
        "DeepDTA model trained on the Davis kinase dataset. Predicts pKd from a drug SMILES and a protein sequence.\n"
    )

api = HfApi(token=HF_TOKEN)
username = api.whoami()["name"]
space_name = "deepdta-davis"
repo_id = username + "/" + space_name

api.create_repo(repo_id=repo_id, repo_type="space", space_sdk="static", exist_ok=True)
api.upload_folder(folder_path=WEB_DIR, repo_id=repo_id, repo_type="space")

permanent_url = ("https://" + username + "-" + space_name + ".hf.space").lower()
print("Uploaded to: https://huggingface.co/spaces/" + repo_id)
print("Permanent link (ready in about a minute):")
print(permanent_url)